# When Marketing Models Try Too Hard

**A Practical Guide to Underfitting, Overfitting, and Building Models That Generalize**

This notebook is the hands-on companion to my article *When Marketing Models Try Too Hard* on the [Marketing Data Science blog](https://blog.marketingdatascience.ai). <!-- TODO: replace with live article URL after publication -->

The article explains the concepts. This notebook lets you watch them happen.

We'll use a synthetic dataset of sales-qualified leads and build three models: one that learns too little, one that learns too much, and one that gets it just right. Along the way we'll demonstrate cross-validation, time-based validation, regularization, feature selection, data leakage, and marketing's small-n problem.

**How to use this notebook:**
- Run the cells in order from top to bottom (Runtime > Run all works too).
- Every random process is seeded, so your numbers will match the article exactly.
- No prior scikit-learn experience needed. The code is meant to be read, not memorized.

*Author: Joe Domaleski | [GitHub](https://github.com/joedom99) | [Marketing Data Science](https://blog.marketingdatascience.ai)*

## 1. Setup

Standard Python data science stack. Everything here comes preinstalled in Google Colab.

One important line: `SEED = 430`. Machine learning involves randomness (which leads end up in training vs. validation, how noise gets generated). Setting a seed makes the randomness repeatable, so the numbers you see match the numbers in the article.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, r2_score

# One seed to rule them all. Change it and watch every number shift slightly.
SEED = 430

# A clean, publication-style look for every chart in this notebook
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25,
    'font.family': 'sans-serif', 'font.size': 11,
    'axes.titlesize': 12, 'figure.dpi': 110
})
BLUE, ORANGE, GREEN, GRAY = "#4C72B0", "#DD8452", "#55A868", "#8C8C8C"

os.makedirs('figures', exist_ok=True)
print("Setup complete.")

## 2. Generate a Synthetic Marketing Dataset

Our running example: **predicting whether a sales-qualified lead becomes a customer.**

We generate 1,000 leads. Six features genuinely influence conversion:

| Feature | Description |
|---|---|
| `company_size` | Number of employees |
| `website_visits` | Visits in the last 90 days |
| `email_engagement` | Open/click engagement score (0 to 1) |
| `webinar_attended` | Attended a webinar (0/1) |
| `demo_requested` | Requested a product demo (0/1) |
| `industry_fit` | Lead is in a target industry (0/1) |

We also add **40 extra columns** (`crm_field_1` through `crm_field_40`) that have *nothing* to do with conversion. Think of them as the junk that comes along in every CRM export: legacy scores, stale custom fields, half-populated attributes. They exist to give an overly eager model something to memorize.

Because this data is synthetic, we know the ground truth. That's the whole point. In the real world you never know which columns matter. Here, we do, so we can see exactly how models fool themselves.

In [ ]:
def make_leads(seed=SEED, n=1000, noise_feats=40):
    """Generate a synthetic dataset of sales-qualified leads."""
    rng = np.random.default_rng(seed)

    # Six features that genuinely drive conversion
    company_size     = rng.lognormal(4.0, 1.0, n)                  # employees
    website_visits   = rng.poisson(8, n) + rng.integers(0, 5, n)   # visits, last 90 days
    email_engagement = np.clip(rng.beta(2, 3, n), 0, 1)            # engagement score
    webinar          = rng.binomial(1, 0.35, n)                    # attended webinar
    demo             = rng.binomial(1, 0.30, n)                    # requested demo
    industry_fit     = rng.binomial(1, 0.50, n)                    # target industry

    # The true (hidden) relationship between behavior and conversion
    z = (1.1 * np.log(company_size) / 2
         + 0.12 * website_visits
         + 2.0 * email_engagement
         + 1.0 * webinar
         + 1.4 * demo
         + 0.8 * industry_fit)
    z = (z - z.mean()) / z.std()
    p = 1 / (1 + np.exp(-5.0 * z))
    y = rng.binomial(1, p)

    # Real customers are never perfectly predictable. Flip 6% of outcomes.
    flip = rng.random(n) < 0.06
    y = np.where(flip, 1 - y, y)

    df = pd.DataFrame({
        'company_size': company_size, 'website_visits': website_visits,
        'email_engagement': email_engagement, 'webinar_attended': webinar,
        'demo_requested': demo, 'industry_fit': industry_fit
    })

    # 40 CRM columns that are pure noise
    for i in range(noise_feats):
        df[f'crm_field_{i+1}'] = rng.normal(0, 1, n)

    df['converted'] = y
    return df

leads = make_leads()
REAL_FEATURES  = ['company_size', 'website_visits', 'email_engagement',
                  'webinar_attended', 'demo_requested', 'industry_fit']
NOISE_FEATURES = [c for c in leads.columns if c.startswith('crm_field')]

print(f"Dataset: {leads.shape[0]} leads, {leads.shape[1]-1} features")
print(f"Conversion rate: {leads['converted'].mean():.1%}")
leads[REAL_FEATURES + ['crm_field_1', 'converted']].head()

One note on metrics before we go further. This dataset converts at roughly 50%, so plain **accuracy** is a fair and easy-to-read metric, and that's what we'll use throughout. Real lead conversion data is usually imbalanced (say, 5% conversion), and in that world you'd reach for AUC, precision, recall, or F1 instead. The concepts in this notebook apply the same way regardless of metric.

## 3. The Goldilocks Principle, Visualized

Before we model the leads dataset, here's the whole article in one picture. We fit three polynomial curves to the same small set of customer data points:

- **Too simple:** a straight line that misses the pattern
- **Just right:** a smooth curve that captures the pattern
- **Too complex:** a curve that chases every individual point

The data points are identical in all three panels. Only the model's flexibility changes.

In [ ]:
rng = np.random.default_rng(7)
x = np.sort(rng.uniform(0, 1, 20))
y_pts = np.sin(2.2 * np.pi * x * 0.8 + 0.3) + 0.5 * x + rng.normal(0, 0.22, len(x))
xx = np.linspace(0, 1, 400)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
for ax, deg, title in zip(axes, [1, 4, 16],
                          ['Underfit (degree 1)', 'Good fit (degree 4)', 'Overfit (degree 16)']):
    poly = np.polynomial.Polynomial.fit(x, y_pts, deg)
    ax.scatter(x, y_pts, s=24, color=GRAY, zorder=3)
    ax.plot(xx, poly(xx), color=BLUE if deg == 4 else ORANGE, lw=2.2)
    ax.set_ylim(-2.0, 2.4)
    ax.set_title(title)
    ax.set_xlabel('Lead engagement score')
axes[0].set_ylabel('Likelihood of conversion')
plt.tight_layout()
plt.savefig('figures/fig1_goldilocks.png', bbox_inches='tight')
plt.show()

The underfit line has learned too little. The overfit curve has learned too much: it passes through nearly every point, but those violent swings between points are pure fiction. Show it a new lead and its prediction is as likely to be driven by noise as by signal.

The middle panel is the goal. Now let's see the same thing happen with a real modeling workflow.

## 4. Training, Validation, and Test Sets

We split our 1,000 leads three ways:

- **Training set (600 leads):** what the model learns from
- **Validation set (200 leads):** used to compare models and catch overfitting
- **Test set (200 leads):** touched exactly once, at the very end, for a final honest check

Why three sets instead of two? Because every time you use the validation set to make a decision, you leak a little information into your choices. The test set stays locked away so at least one number remains completely untouched by your decisions.

In [ ]:
X = leads.drop(columns='converted')
y = leads['converted']

# First carve off the test set (20%), then split the rest into train (60%) and validation (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=SEED, stratify=y_temp)

def accuracy(model, X_data, y_data):
    """Accuracy as a percentage."""
    return accuracy_score(y_data, model.predict(X_data)) * 100

print(f"Training:   {len(X_train)} leads")
print(f"Validation: {len(X_val)} leads")
print(f"Test:       {len(X_test)} leads (locked away until the end)")

## 5. Underfitting: Learning Too Little

Our underfit model uses exactly one feature: `demo_requested`. A demo request is a real buying signal, so this model isn't useless. It's just far too simple to capture how six different behaviors combine into a conversion decision.

In [ ]:
underfit_model = LogisticRegression(max_iter=2000)
underfit_model.fit(X_train[['demo_requested']], y_train)

under_train = accuracy(underfit_model, X_train[['demo_requested']], y_train)
under_val   = accuracy(underfit_model, X_val[['demo_requested']], y_val)
print(f"Underfit model | Training: {under_train:.0f}%  Validation: {under_val:.0f}%")

Both numbers are mediocre, and they're mediocre *together*. That's the signature of underfitting: the model performs poorly on data it has already seen. It hasn't memorized noise. It just hasn't learned enough signal.

## 6. Good Fit: Learning the Right Amount

Now we use all six real features with the same simple algorithm, logistic regression. (We standardize the features first so they're on comparable scales, a standard practice for linear models.)

In [ ]:
goodfit_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
goodfit_model.fit(X_train[REAL_FEATURES], y_train)

good_train = accuracy(goodfit_model, X_train[REAL_FEATURES], y_train)
good_val   = accuracy(goodfit_model, X_val[REAL_FEATURES], y_val)
print(f"Good fit model | Training: {good_train:.0f}%  Validation: {good_val:.0f}%")

Training and validation performance are high *and close together*. The model has found patterns that hold up on leads it has never seen. This is what generalization looks like.

## 7. Overfitting: Learning Too Much

For the overfit model we hand a fully grown decision tree all 46 columns, including the 40 junk CRM fields, and let it grow with no depth limit. The tree will keep splitting until it can classify every single training lead correctly, even if that means building rules out of pure noise.

In [ ]:
ALL_FEATURES = REAL_FEATURES + NOISE_FEATURES

overfit_model = DecisionTreeClassifier(random_state=SEED)  # no depth limit
overfit_model.fit(X_train[ALL_FEATURES], y_train)

over_train = accuracy(overfit_model, X_train[ALL_FEATURES], y_train)
over_val   = accuracy(overfit_model, X_val[ALL_FEATURES], y_val)
print(f"Overfit model | Training: {over_train:.0f}%  Validation: {over_val:.0f}%")

In [ ]:
summary = pd.DataFrame({
    'Model': ['Underfit', 'Good Fit', 'Overfit'],
    'Training Accuracy': [f"{under_train:.0f}%", f"{good_train:.0f}%", f"{over_train:.0f}%"],
    'Validation Accuracy': [f"{under_val:.0f}%", f"{good_val:.0f}%", f"{over_val:.0f}%"]
})
summary

This is the table from the article. Look at the overfit row carefully.

The overfit model posts a *perfect* training score and the *worst* validation score of the three. It's worse than the model that only knew about demo requests. All that extra complexity didn't just fail to help. It actively hurt, because the tree spent its capacity memorizing noise in the 40 junk columns.

If someone showed you only the training accuracy, the overfit model would look like the obvious winner. That's exactly how impressive-looking models fail in production.

## 8. Watching Overfitting Begin

Underfitting and overfitting aren't two separate diseases. They're the two ends of a single dial: **model complexity**. Let's turn that dial slowly and watch what happens.

We'll refit the decision tree at every depth from 1 to 15 and track training and validation accuracy at each step.

In [ ]:
depths = range(1, 16)
train_curve, val_curve = [], []

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    tree.fit(X_train[ALL_FEATURES], y_train)
    train_curve.append(accuracy(tree, X_train[ALL_FEATURES], y_train))
    val_curve.append(accuracy(tree, X_val[ALL_FEATURES], y_val))

best_depth = list(depths)[int(np.argmax(val_curve))]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(depths, train_curve, 'o-', color=BLUE, lw=2, label='Training accuracy')
ax.plot(depths, val_curve, 'o-', color=ORANGE, lw=2, label='Validation accuracy')
ax.axvline(best_depth, color=GRAY, ls='--', lw=1.2)
ax.annotate('overfitting begins', xy=(best_depth, 76), xytext=(best_depth + 2.2, 82),
            arrowprops=dict(arrowstyle='->', color='black', lw=1))
ax.set_xlabel('Tree depth (model complexity)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Training vs. Validation Accuracy as Complexity Increases')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('figures/fig2_complexity.png', bbox_inches='tight')
plt.show()

print(f"Validation accuracy peaks at depth {best_depth}, then declines while training accuracy keeps climbing.")

The two curves tell the whole story. Training accuracy climbs relentlessly toward 100% because a deeper tree can always memorize more. Validation accuracy rises at first (the model is learning real signal), peaks, and then *falls* as the extra capacity gets spent on noise.

Everything to the right of the dashed line is overfitting. More complexity, worse predictions.

Notice the validation curve is a bit jagged. That's because it comes from a single 200-lead validation sample, and 200 leads is a noisy yardstick. Which brings us to cross-validation.

## 9. Cross-Validation: A More Reliable Yardstick

A single train/validation split gives you one estimate of generalization. But that estimate depends on which leads happened to land in the validation set. Get a lucky split and your model looks better than it is. Unlucky, worse.

To see how much luck is involved, let's evaluate our good-fit model on 20 *different* random splits of the same data.

In [ ]:
split_scores = []
for s in range(20):
    Xa, Xb, ya, yb = train_test_split(X_temp[REAL_FEATURES], y_temp,
                                      test_size=0.25, random_state=s, stratify=y_temp)
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xa, ya)
    split_scores.append(accuracy(m, Xb, yb))

split_scores = np.array(split_scores)
print(f"Same model, same data, 20 different random splits:")
print(f"  Lowest estimate:  {split_scores.min():.1f}%")
print(f"  Highest estimate: {split_scores.max():.1f}%")

In [ ]:
# k-fold cross-validation: every lead takes a turn in the validation set
cv_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
cv_scores = cross_val_score(cv_model, X_temp[REAL_FEATURES], y_temp,
                            cv=5, scoring='accuracy') * 100

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.scatter(split_scores, np.zeros_like(split_scores), s=60, alpha=0.5,
           color=ORANGE, label='20 single-split estimates')
ax.axvline(cv_scores.mean(), color=BLUE, lw=2.5,
           label=f'5-fold CV estimate ({cv_scores.mean():.1f}%)')
ax.set_yticks([])
ax.set_xlabel('Estimated validation accuracy (%)')
ax.set_title('Single Splits Are Noisy. Cross-Validation Averages the Noise Away.')
ax.legend(frameon=False, loc='upper left')
plt.tight_layout()
plt.savefig('figures/fig3_cv_variability.png', bbox_inches='tight')
plt.show()

print(f"5-fold cross-validation: {cv_scores.mean():.1f}% (+/- {cv_scores.std():.1f})")

Depending on the split you happened to draw, you might believe your model is a 78% model or an 88% model. That's a huge range for a decision as important as "should we deploy this?"

**k-fold cross-validation** fixes this by rotating: split the data into k folds (here, 5), train on four of them, validate on the fifth, and repeat until every fold has taken a turn as the validation set. Averaging the five estimates gives a far more stable read on generalization, and you get a standard deviation for free.

## 10. Time-Based Validation: Marketing Data Has a Clock

Everything so far treated leads as interchangeable rows. But a lot of marketing data is **chronological**: weekly sales, campaign performance, marketing mix data. And time changes the validation rules.

Here's why. If you randomly shuffle three years of weekly data into folds, the model gets to train on December while being tested on the November that came *before* it. It can quietly interpolate trends and seasonality it should have had to forecast. The result is an optimistic score you'll never see again in the real world, where models only get to predict *forward*.

Let's build a small marketing mix style dataset (156 weeks, the classic "three years of weekly data") and compare the two validation strategies. We'll reuse this dataset later for the small-n problem.

In [ ]:
def make_mmm(seed=SEED, weeks=156, n_candidates=10):
    """Synthetic weekly marketing mix data: sales driven by 4 channels,
    seasonality, and an upward trend, plus candidate variables that do nothing."""
    rng = np.random.default_rng(seed)
    t = np.arange(weeks)

    tv     = np.abs(rng.normal(50, 20, weeks))   # weekly spend, $K
    search = np.abs(rng.normal(30, 10, weeks))
    social = np.abs(rng.normal(20, 8, weeks))
    email  = np.abs(rng.normal(10, 4, weeks))
    seasonality = 10 * np.sin(2 * np.pi * t / 52)

    sales = (200 + 0.8 * tv + 1.2 * search + 0.6 * social + 1.5 * email
             + seasonality + 0.5 * t + rng.normal(0, 18, weeks))

    df = pd.DataFrame({'week': t, 'tv': tv, 'search': search,
                       'social': social, 'email': email})
    # Candidate variables marketers often want to include, none of which
    # actually drive sales in this simulation
    for i in range(n_candidates):
        df[f'candidate_{i+1}'] = np.abs(rng.normal(25, 10, weeks))
    df['sales'] = sales
    return df

mmm = make_mmm()
X_mmm = mmm.drop(columns='sales')
y_mmm = mmm['sales']
print(f"MMM dataset: {len(mmm)} weekly observations, {X_mmm.shape[1]} variables")
mmm.head(3)

In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=SEED)

# Strategy 1: shuffled 5-fold CV (ignores time)
shuffled_cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
r2_shuffled = cross_val_score(rf, X_mmm, y_mmm, cv=shuffled_cv, scoring='r2').mean()

# Strategy 2: chronological split (train on the first 125 weeks, validate on the last 31)
cut = int(len(mmm) * 0.8)
rf.fit(X_mmm[:cut], y_mmm[:cut])
r2_chrono = r2_score(y_mmm[cut:], rf.predict(X_mmm[cut:]))

fig, ax = plt.subplots(figsize=(7, 3.8))
bars = ax.bar(['Shuffled 5-fold CV\n(leaks the future)', 'Chronological split\n(honest forecast)'],
              [r2_shuffled, r2_chrono], color=[ORANGE, BLUE], width=0.5)
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('R² on held-out data')
ax.set_title('The Same Model Looks Very Different Depending on How You Validate')
for bar, v in zip(bars, [r2_shuffled, r2_chrono]):
    ax.text(bar.get_x() + bar.get_width()/2, v + (0.02 if v > 0 else -0.05),
            f'{v:.2f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig6_time_validation.png', bbox_inches='tight')
plt.show()

print(f"Shuffled CV says R² = {r2_shuffled:.2f}. Forecasting forward in time says R² = {r2_chrono:.2f}.")

Same model. Same data. Shuffled cross-validation claims a respectable R². The honest chronological split reveals the model can barely forecast at all (an R² near or below zero means it does no better than just predicting the historical average).

The shuffled version scored well because it could *interpolate*: with weeks 51 and 53 in the training set, "predicting" week 52 is easy. Real marketing models never get that luxury. Next quarter hasn't happened yet.

**Rule of thumb:** if your data has a timestamp, your validation should respect it. Train on January through September, validate on October through December, test on next quarter. This matters enormously for marketing mix modeling and forecasting.

## 11. Regularization: Built-In Skepticism

So far we've fought overfitting by choosing simpler models. **Regularization** takes a different approach: keep the flexible model, but add a penalty for large coefficients. The model has to *earn* every coefficient by improving fit enough to outweigh the penalty. Weak, noisy patterns don't make the cut.

The two classic flavors:
- **Ridge (L2):** shrinks all coefficients toward zero, but rarely all the way
- **Lasso (L1):** shrinks coefficients and can push the useless ones to *exactly* zero, performing automatic feature selection

If you read my [Bayesian statistics article](https://blog.marketingdatascience.ai), this idea will feel familiar: regularization is mathematically related to placing skeptical prior beliefs on the coefficients. The penalty says "I don't believe this variable matters until the data proves otherwise."

Let's give logistic regression all 46 features, with and without a Lasso-style (L1) penalty, and look at what happens to the coefficients.

In [ ]:
scaler_all = StandardScaler().fit(X_train[ALL_FEATURES])
Xtr_s = scaler_all.transform(X_train[ALL_FEATURES])
Xva_s = scaler_all.transform(X_val[ALL_FEATURES])

# Essentially unregularized: a huge C means a tiny penalty
unreg = LogisticRegression(C=1e6, max_iter=5000).fit(Xtr_s, y_train)

# L1 (Lasso-style) regularization. Note: for classification we use
# LogisticRegression with penalty='l1' rather than the Lasso regressor class.
l1 = LogisticRegression(C=0.1, penalty='l1', solver='liblinear').fit(Xtr_s, y_train)

for name, m in [('Unregularized (46 features)', unreg), ('L1 regularized', l1)]:
    kept = int((np.abs(m.coef_) > 1e-6).sum())
    tr = accuracy_score(y_train, m.predict(Xtr_s)) * 100
    va = accuracy_score(y_val, m.predict(Xva_s)) * 100
    print(f"{name:32s} Train: {tr:.1f}%  Validation: {va:.1f}%  Nonzero coefficients: {kept}/46")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
xpos = np.arange(len(ALL_FEATURES))
colors = [BLUE] * len(REAL_FEATURES) + [GRAY] * len(NOISE_FEATURES)

for ax, model, title in [(axes[0], unreg, 'Unregularized: noise columns get real-looking coefficients'),
                         (axes[1], l1, 'L1 regularized: most noise columns are forced to exactly zero')]:
    ax.bar(xpos, model.coef_[0], color=colors, width=0.8)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(title, loc='left', fontsize=11)
    ax.set_ylabel('Coefficient')

axes[1].set_xticks([2.5, 25])
axes[1].set_xticklabels(['6 real features (blue)', '40 noise CRM fields (gray)'])
plt.tight_layout()
plt.savefig('figures/fig4_regularization.png', bbox_inches='tight')
plt.show()

In the top panel, the unregularized model hands out nonzero coefficients to all 40 junk CRM fields. Every one of those coefficients is a small piece of memorized noise, and together they drag validation accuracy down.

In the bottom panel, the L1 penalty zeroes out most of the junk while keeping the real features. Validation accuracy improves, and the model got *simpler* automatically.

One caution: regularization strength is itself a Goldilocks dial. Too little penalty and noise survives. Too much and you start crushing real signal, which is just underfitting with extra steps. We'll see that happen in the small-n section. (In practice you choose the penalty strength by, you guessed it, cross-validation.)

## 12. Feature Selection: Sometimes Less Really Is More

Regularization removed junk features automatically. But there's a more direct lesson hiding in this dataset: what if we simply *don't include* the junk in the first place?

Compare the same logistic regression trained on all 46 columns vs. only the 6 real ones.

In [ ]:
results = pd.DataFrame({
    'Feature set': ['All 46 features (6 real + 40 noise)', 'Only the 6 real features'],
    'Training accuracy': [f"{accuracy_score(y_train, unreg.predict(Xtr_s))*100:.1f}%",
                          f"{good_train:.1f}%"],
    'Validation accuracy': [f"{accuracy_score(y_val, unreg.predict(Xva_s))*100:.1f}%",
                            f"{good_val:.1f}%"]
})
results

Same algorithm, same training data, same conversion outcomes. Removing 40 variables *improved* validation accuracy by about five points.

This runs against a deep marketing instinct: "we have the data, so we should use it." But every irrelevant variable you add gives the model another opportunity to find a coincidental pattern. In practice, watch for three kinds of removable variables:

- **Noisy predictors:** variables with no plausible connection to the outcome
- **Redundant variables:** two fields measuring nearly the same thing
- **Multicollinearity:** predictors so correlated the model can't tell their effects apart, which makes coefficients unstable and hard to interpret

A shorter feature list is easier to explain to stakeholders, too. That matters more than data scientists like to admit.

## 13. Data Leakage: The Miracle Model That Isn't

Now for the most dangerous failure mode in this notebook, because it produces models that look *spectacular*.

**Data leakage** happens when a feature contains information about the outcome that wouldn't be available at prediction time. The model isn't predicting anymore. It's peeking at the answer.

The obvious version is including the target itself. Nobody does that. The dangerous version is subtler. Consider a field like `contract_sent`. It sits innocently in your CRM export right next to `demo_requested`. But contracts get sent *because* a lead is converting. It's a consequence of the outcome dressed up as a predictor.

Let's add it and watch what happens. We'll generate it realistically: 90% of converting leads got a contract, and 5% of non-converting leads did too (deals that fell through at the last minute).

In [ ]:
rng = np.random.default_rng(SEED)
leaky = leads.copy()
leaky['contract_sent'] = np.where(
    leaky['converted'] == 1,
    (rng.random(len(leaky)) < 0.90).astype(int),   # 90% of converters got a contract
    (rng.random(len(leaky)) < 0.05).astype(int)    # 5% of non-converters did too
)

Xl = leaky.drop(columns='converted')
yl = leaky['converted']
Xl_temp, Xl_test, yl_temp, yl_test = train_test_split(Xl, yl, test_size=0.20,
                                                      random_state=SEED, stratify=yl)
Xl_train, Xl_val, yl_train, yl_val = train_test_split(Xl_temp, yl_temp, test_size=0.25,
                                                      random_state=SEED, stratify=yl_temp)

LEAKY_FEATURES = REAL_FEATURES + ['contract_sent']
leaky_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
leaky_model.fit(Xl_train[LEAKY_FEATURES], yl_train)

leak_train = accuracy(leaky_model, Xl_train[LEAKY_FEATURES], yl_train)
leak_val   = accuracy(leaky_model, Xl_val[LEAKY_FEATURES], yl_val)

print(f"Clean model:  Training {good_train:.0f}%   Validation {good_val:.0f}%")
print(f"Leaky model:  Training {leak_train:.0f}%   Validation {leak_val:.0f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
labels = ['Clean model\n(6 real features)', 'Leaky model\n(+ contract_sent)']
tr_vals, va_vals = [good_train, leak_train], [good_val, leak_val]
xpos = np.arange(2)
ax.bar(xpos - 0.18, tr_vals, 0.36, color=BLUE, label='Training')
ax.bar(xpos + 0.18, va_vals, 0.36, color=ORANGE, label='Validation')
for i, (t, v) in enumerate(zip(tr_vals, va_vals)):
    ax.text(i - 0.18, t + 0.6, f'{t:.0f}%', ha='center', fontweight='bold')
    ax.text(i + 0.18, v + 0.6, f'{v:.0f}%', ha='center', fontweight='bold')
ax.set_xticks(xpos); ax.set_xticklabels(labels)
ax.set_ylim(60, 102); ax.set_ylabel('Accuracy (%)')
ax.set_title('Leakage Makes a Model Look Brilliant on Paper')
ax.legend(frameon=False, loc='lower right')
plt.tight_layout()
plt.savefig('figures/fig5_leakage.png', bbox_inches='tight')
plt.show()

Validation accuracy jumps from the mid-80s to the mid-90s. Notice what makes this so dangerous: **validation didn't catch it.** The leak is in both the training and validation data, so every number in your workflow agrees the model is brilliant. Cross-validation would agree too.

Then you deploy it. New leads haven't been sent contracts yet (that's the whole point of scoring them early), so `contract_sent` is zero for everyone, and the model's real-world performance collapses.

The tell in this output is that the jump came from adding one field, and the field is suspiciously well-timed. Whenever a single feature produces a dramatic improvement, ask the leakage question: **"Would I actually know this value at the moment I need the prediction?"** For `contract_sent` the answer is no. Drop it, and the honest 84% model is what you actually have.

Leakage can also enter through sloppy workflow rather than bad features: fitting your scaler or selecting features on the full dataset before splitting, for example. The discipline is the same. Nothing computed from validation or test data should ever touch the training process.

## 14. Marketing's Small-n Problem

Image recognition models train on millions of examples. Marketing mix models train on... three years of weekly data. That's the 156-row dataset we built in Section 10.

Now count what marketers want to estimate from those 156 rows: TV, paid search, paid social, email, pricing, promotions, holidays, seasonality, competitor activity, economic conditions. The list of candidate variables grows fast while the row count stays fixed. That ratio, many variables to few observations, is a structural invitation to overfit.

Our synthetic MMM data has 4 channels that truly drive sales, plus seasonality and trend, plus 10 candidate variables that do *nothing*. Watch what plain OLS regression does with them, and what Ridge regularization does differently. (Since this is a regression problem, we can use the actual `Ridge` and `Lasso` classes here.)

In [ ]:
features_mmm = [c for c in X_mmm.columns if c != 'week']
scaler_mmm = StandardScaler().fit(X_mmm[features_mmm])
Xm_s = scaler_mmm.transform(X_mmm[features_mmm])

ols_mmm   = LinearRegression().fit(Xm_s, y_mmm)
ridge_mmm = Ridge(alpha=50).fit(Xm_s, y_mmm)

coef_table = pd.DataFrame({
    'Variable': features_mmm,
    'OLS coefficient': ols_mmm.coef_.round(2),
    'Ridge coefficient': ridge_mmm.coef_.round(2),
    'Truly drives sales?': ['Yes'] * 4 + ['No'] * 10
})
coef_table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ypos = np.arange(len(features_mmm))
colors = [BLUE] * 4 + [GRAY] * 10
ax.barh(ypos - 0.18, ols_mmm.coef_, 0.36, color=colors, label='OLS')
ax.barh(ypos + 0.18, ridge_mmm.coef_, 0.36, color=colors, alpha=0.45, label='Ridge (alpha=50)')
ax.axvline(0, color='black', lw=0.8)
ax.set_yticks(ypos); ax.set_yticklabels(features_mmm)
ax.invert_yaxis()
ax.set_xlabel('Standardized coefficient')
ax.set_title('156 Weeks, 14 Variables: OLS Finds "Effects" That Do Not Exist')
ax.legend(frameon=False, loc='lower right')
plt.tight_layout()
plt.savefig('figures/fig7_smalln.png', bbox_inches='tight')
plt.show()

worst = coef_table.iloc[coef_table['OLS coefficient'].abs()[4:].idxmax()]
print(f"Largest phantom effect under OLS: {worst['Variable']} "
      f"({worst['OLS coefficient']}), bigger in magnitude than email's real effect.")

Look at `candidate_2`. It's random noise. It has zero true effect on sales. Yet OLS assigns it a coefficient larger in magnitude than email, a channel that *genuinely* drives revenue. Take this model to a budget meeting and it will confidently tell you a phantom variable is suppressing sales.

With only 156 observations and 14 variables, OLS has enough freedom to fit coincidences. Ridge shrinks the phantom coefficients toward zero while keeping the real channels prominent, at the small cost of slightly shrinking the real effects too.

And remember the Goldilocks warning from Section 11: crank the Lasso penalty high enough on this data and it starts zeroing out *real* channels like social and email along with the noise. Regularization strength is a dial, not a switch.

This is exactly why modern MMM frameworks build regularization into their foundations. Robyn uses ridge regression at its core, and Meridian's Bayesian priors serve the same skeptical function. With marketing's small-n reality, unregularized MMM isn't a modeling choice, it's a coin flip. If you want the deeper story on those frameworks, see my MMM series on the blog.

## 15. The Final Honest Check

One promise left to keep. Back in Section 4 we locked away 200 leads as a test set. We've made all our modeling decisions, so we're allowed to touch it exactly once: a final check that our chosen model (the good-fit logistic regression) holds up on data that influenced nothing.

In [ ]:
final_test = accuracy(goodfit_model, X_test[REAL_FEATURES], y_test)
print(f"Good fit model | Training: {good_train:.0f}%   Validation: {good_val:.0f}%   Test: {final_test:.0f}%")

Test accuracy lands right beside the validation estimate. No surprises, which is exactly what you want. The model learned real patterns, and those patterns held on completely untouched data.

## Key Takeaways

1. **Generalization is the goal.** Training accuracy tells you what a model memorized. Validation and test accuracy tell you what it learned.
2. **Complexity is a dial with a sweet spot.** Too simple misses the signal (underfitting). Too complex memorizes the noise (overfitting). The overfit model in this notebook scored 100% on training data and *worst of all* on new leads.
3. **Cross-validation beats a single split.** One random split can mislead you by 10 points. Averaging across folds gives a stable estimate.
4. **If your data has a timestamp, validate forward in time.** Shuffled CV claimed R² of about 0.5 on our weekly data. Honest chronological validation revealed the model couldn't forecast at all.
5. **Regularization is built-in skepticism.** Ridge and Lasso force models to earn their coefficients. That's why Robyn and Meridian have it baked in.
6. **Fewer features often means better predictions.** Removing 40 junk columns improved validation accuracy by five points.
7. **Data leakage makes bad models look brilliant.** Always ask: would I know this value at prediction time?
8. **Marketing is a small-n discipline.** 156 weekly observations can't support unlimited variables. Respect the ratio.

The purpose of machine learning isn't to build the most sophisticated model. It's to make better marketing decisions. The best models aren't the ones that explain yesterday perfectly. They're the ones that keep working when tomorrow's customers behave just a little differently than yesterday's.

---

*If you found this useful, read the full article at [Marketing Data Science](https://blog.marketingdatascience.ai) and explore the code on [GitHub](https://github.com/joedom99).* <!-- TODO: replace with final article + repo URLs -->